In [1]:
using JuMP
using HiGHS
# using EzXML
using Plots
# using Ipopt
# using Cbc
import MultiObjectiveAlgorithms as MOA
import MathOptInterface as MOI
using Printf
using PrettyTables

In [2]:
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7

7

In [3]:
function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
        Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
        Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
        Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
        Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
        w = parse.(Int, split(lines[idx])); idx += 1
        b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        v = parse.(Int, split(lines[idx])); idx += 1
        V = parse.(Int, split(lines[idx])); idx += 1

        B_star = Array{Int, 3}(undef, S, P, length(D_star))
        for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        C_star = Array{Int, 3}(undef, T, P, length(D_star))
        for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        if validate==true
                # input validation


        end

        return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star
end

read_input (generic function with 1 method)

In [4]:
function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

ρ (generic function with 1 method)

In [5]:
# Get input parameters from .txt file
input_files_folder = "input"
input_file_name = "input_smaller.txt"
# input_file_name = "input_current.txt"
file_path = input_files_folder * "/" * input_file_name
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

(28, [1, 3, 5, 8, 10, 12, 15, 17, 19, 22, 24, 26], [14, 28], 10, SubString{String}["Player1", "Player2", "Player3", "Player4", "Player5", "Player6", "Player7", "Player8", "Player9", "Player10"], 5, SubString{String}["Dribling", "Passing", "Shooting", "Defending", "Rebounding"], 3, SubString{String}["Tactic1", "Tactic2", "Tactic3"], 8, SubString{String}["Exercise1", "Exercise2", "Exercise3", "Exercise4", "Exercise5", "Exercise6", "Exercise7", "Exercise8"], [100, 90, 110, 85, 100, 100, 90, 110, 85, 100], [[5, 6, 7, 8, 6, 5, 6, 7, 8, 6], [8, 9, 5, 6, 7, 8, 9, 5, 6, 7], [4, 5, 9, 8, 6, 4, 5, 9, 8, 6], [5, 6, 7, 8, 6, 5, 6, 7, 8, 6], [8, 9, 5, 6, 7, 8, 9, 5, 6, 7]], [[7, 6, 4, 6, 6, 7, 6, 4, 6, 6], [8, 5, 5, 8, 7, 8, 5, 5, 8, 7], [4, 5, 4, 8, 6, 4, 5, 4, 8, 6]], [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0], [1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1], [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1,

In [6]:
# Define model
model = Model(HiGHS.Optimizer)
# model = Model(() -> MOA.Optimizer(
#         optimizer_with_attributes(
#         HiGHS.Optimizer,
#         "output_flag" => true,
#         "mip_rel_gap" => 0.05,
#         "threads" => 8
#         )
# ))

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [7]:
# Define variables
@variable(model, x[1:E, 1:length(D_)] >= 0, Int)

8×12 Matrix{VariableRef}:
 x[1,1]  x[1,2]  x[1,3]  x[1,4]  x[1,5]  …  x[1,9]  x[1,10]  x[1,11]  x[1,12]
 x[2,1]  x[2,2]  x[2,3]  x[2,4]  x[2,5]     x[2,9]  x[2,10]  x[2,11]  x[2,12]
 x[3,1]  x[3,2]  x[3,3]  x[3,4]  x[3,5]     x[3,9]  x[3,10]  x[3,11]  x[3,12]
 x[4,1]  x[4,2]  x[4,3]  x[4,4]  x[4,5]     x[4,9]  x[4,10]  x[4,11]  x[4,12]
 x[5,1]  x[5,2]  x[5,3]  x[5,4]  x[5,5]     x[5,9]  x[5,10]  x[5,11]  x[5,12]
 x[6,1]  x[6,2]  x[6,3]  x[6,4]  x[6,5]  …  x[6,9]  x[6,10]  x[6,11]  x[6,12]
 x[7,1]  x[7,2]  x[7,3]  x[7,4]  x[7,5]     x[7,9]  x[7,10]  x[7,11]  x[7,12]
 x[8,1]  x[8,2]  x[8,3]  x[8,4]  x[8,5]     x[8,9]  x[8,10]  x[8,11]  x[8,12]

In [8]:
@variable(model, y[1:S, 1:P, 1:length(D_star)], Bin)

5×10×2 Array{VariableRef, 3}:
[:, :, 1] =
 y[1,1,1]  y[1,2,1]  y[1,3,1]  y[1,4,1]  …  y[1,8,1]  y[1,9,1]  y[1,10,1]
 y[2,1,1]  y[2,2,1]  y[2,3,1]  y[2,4,1]     y[2,8,1]  y[2,9,1]  y[2,10,1]
 y[3,1,1]  y[3,2,1]  y[3,3,1]  y[3,4,1]     y[3,8,1]  y[3,9,1]  y[3,10,1]
 y[4,1,1]  y[4,2,1]  y[4,3,1]  y[4,4,1]     y[4,8,1]  y[4,9,1]  y[4,10,1]
 y[5,1,1]  y[5,2,1]  y[5,3,1]  y[5,4,1]     y[5,8,1]  y[5,9,1]  y[5,10,1]

[:, :, 2] =
 y[1,1,2]  y[1,2,2]  y[1,3,2]  y[1,4,2]  …  y[1,8,2]  y[1,9,2]  y[1,10,2]
 y[2,1,2]  y[2,2,2]  y[2,3,2]  y[2,4,2]     y[2,8,2]  y[2,9,2]  y[2,10,2]
 y[3,1,2]  y[3,2,2]  y[3,3,2]  y[3,4,2]     y[3,8,2]  y[3,9,2]  y[3,10,2]
 y[4,1,2]  y[4,2,2]  y[4,3,2]  y[4,4,2]     y[4,8,2]  y[4,9,2]  y[4,10,2]
 y[5,1,2]  y[5,2,2]  y[5,3,2]  y[5,4,2]     y[5,8,2]  y[5,9,2]  y[5,10,2]

In [9]:
@variable(model, z[1:T, 1:P, 1:length(D_star)], Bin)

3×10×2 Array{VariableRef, 3}:
[:, :, 1] =
 z[1,1,1]  z[1,2,1]  z[1,3,1]  z[1,4,1]  …  z[1,8,1]  z[1,9,1]  z[1,10,1]
 z[2,1,1]  z[2,2,1]  z[2,3,1]  z[2,4,1]     z[2,8,1]  z[2,9,1]  z[2,10,1]
 z[3,1,1]  z[3,2,1]  z[3,3,1]  z[3,4,1]     z[3,8,1]  z[3,9,1]  z[3,10,1]

[:, :, 2] =
 z[1,1,2]  z[1,2,2]  z[1,3,2]  z[1,4,2]  …  z[1,8,2]  z[1,9,2]  z[1,10,2]
 z[2,1,2]  z[2,2,2]  z[2,3,2]  z[2,4,2]     z[2,8,2]  z[2,9,2]  z[2,10,2]
 z[3,1,2]  z[3,2,2]  z[3,3,2]  z[3,4,2]     z[3,8,2]  z[3,9,2]  z[3,10,2]

In [10]:
@variable(model, Z[1:T, 1:length(D_star)], Bin)

3×2 Matrix{VariableRef}:
 Z[1,1]  Z[1,2]
 Z[2,1]  Z[2,2]
 Z[3,1]  Z[3,2]

In [11]:
@variable(model, A_min[1:length(D_star)])

2-element Vector{VariableRef}:
 A_min[1]
 A_min[2]

In [12]:
# Вспомогательные функции и компоненты целевой функции
function A(p::Int, d::Int)
	result = a0[p]
	result += sum(δ1(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	result -= sum(δ2(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function A_avg(d::Int)
	return sum(A(p, d) for p in 1:P)/P
end

α = 0.5
function A()
	return sum(α*A_min[d_]+(1-α)*A_avg(D_star[d_]) for d_ in 1:length(D_star))
end

function B(s::Int, p::Int, d::Int)
	result = b0[s][p]
	result += sum(h[p][d_]*sum(b[s][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function B()
	return sum(sum(sum(β(s, p, d_)*y[s, p, d_] for s in 1:S) for p in 1:P) for d_ in 1:length(D_star))
end

function C(t::Int, p::Int, d::Int)
	result = c0[t][p]
	result += sum(h[p][d_]*sum(c[t][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
end

function C()
	return sum(sum(γ(t, d_)*Z[t, d_] for t in 1:T) for d_ in 1:length(D_star))
end

C (generic function with 2 methods)

In [13]:
# Define constraints
@constraint(model, [d_=1:length(D_)], sum(v[e]*x[e, d_] for e in 1:E) <= V[d_])

12-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 2 x[1,1] + 3 x[2,1] + 4 x[3,1] + 2 x[4,1] + x[5,1] + x[6,1] + 3 x[7,1] + 4 x[8,1] <= 9
 2 x[1,2] + 3 x[2,2] + 4 x[3,2] + 2 x[4,2] + x[5,2] + x[6,2] + 3 x[7,2] + 4 x[8,2] <= 9
 2 x[1,3] + 3 x[2,3] + 4 x[3,3] + 2 x[4,3] + x[5,3] + x[6,3] + 3 x[7,3] + 4 x[8,3] <= 9
 2 x[1,4] + 3 x[2,4] + 4 x[3,4] + 2 x[4,4] + x[5,4] + x[6,4] + 3 x[7,4] + 4 x[8,4] <= 9
 2 x[1,5] + 3 x[2,5] + 4 x[3,5] + 2 x[4,5] + x[5,5] + x[6,5] + 3 x[7,5] + 4 x[8,5] <= 9
 2 x[1,6] + 3 x[2,6] + 4 x[3,6] + 2 x[4,6] + x[5,6] + x[6,6] + 3 x[7,6] + 4 x[8,6] <= 9
 2 x[1,7] + 3 x[2,7] + 4 x[3,7] + 2 x[4,7] + x[5,7] + x[6,7] + 3 x[7,7] + 4 x[8,7] <= 9
 2 x[1,8] + 3 x[2,8] + 4 x[3,8] + 2 x[4,8] + x[5,8] + x[6,8] + 3 x[7,8] + 4 x[8,8] <= 9
 2 x[1,9] + 3 x[2,9] + 4 x[3,9] + 2 x[4,9] + x[5,9] + x[6,9] + 3 x[7,9] + 4 x[8,9] <= 9
 2 x[1,10] + 3 x[2,10] + 4 x[3,10] +

In [14]:
@constraint(model, [s=1:S, p=1:P, d_=1:length(D_star)], B(s, p, D_star[d_]) + B_star[s, p, d_]*(1-y[s, p, d_]) >= B_star[s, p, d_])

5×10×2 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 5 x[1,4] + 5 x[2,4] + 4 x[3,4] + 3 x[4,4] + 3 x[5,4] + 3 x[6,4] + 4 x[7,4] + 3 x[8,4] + 5 x[1,5] + 5 x[2,5] + 4 x[3,5] + 3 x[4,5] + 3 x[5,5] + 3 x[6,5] + 4 x[7,5] + 3 x[8,5] + 5 x[1,6] + 5 x[2,6] + 4 x[3,6] + 3 x[4,6] + 3 x[5,6] + 3 x[6,6] + 4 x[7,6] + 3 x[8,6] - 10 y[1,1,1] >= -5  …  5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 

In [15]:
@constraint(model, [t=1:T, p=1:P, d_=1:length(D_star)], C(t, p, D_star[d_]) + C_star[t, p, d_]*(1-z[t, p, d_]) >= C_star[t, p, d_])

3×10×2 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 5 x[1,4] + 5 x[2,4] + 4 x[3,4] + 3 x[4,4] + 3 x[5,4] + 3 x[6,4] + 4 x[7,4] + 3 x[8,4] + 5 x[1,5] + 5 x[2,5] + 4 x[3,5] + 3 x[4,5] + 3 x[5,5] + 3 x[6,5] + 4 x[7,5] + 3 x[8,5] + 5 x[1,6] + 5 x[2,6] + 4 x[3,6] + 3 x[4,6] + 3 x[5,6] + 3 x[6,6] + 4 x[7,6] + 3 x[8,6] - 10 z[1,1,1] >= -7  …  5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 

In [16]:
@constraint(model, [t=1:T, d_=1:length(D_star)], sum(z[t, p, d_] for p in 1:P) >= ρ(t)*Z[t, d_])

3×2 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}}:
 z[1,1,1] + z[1,2,1] + z[1,3,1] + z[1,4,1] + z[1,5,1] + z[1,6,1] + z[1,7,1] + z[1,8,1] + z[1,9,1] + z[1,10,1] - 2 Z[1,1] >= 0  …  z[1,1,2] + z[1,2,2] + z[1,3,2] + z[1,4,2] + z[1,5,2] + z[1,6,2] + z[1,7,2] + z[1,8,2] + z[1,9,2] + z[1,10,2] - 2 Z[1,2] >= 0
 z[2,1,1] + z[2,2,1] + z[2,3,1] + z[2,4,1] + z[2,5,1] + z[2,6,1] + z[2,7,1] + z[2,8,1] + z[2,9,1] + z[2,10,1] - 2 Z[2,1] >= 0     z[2,1,2] + z[2,2,2] + z[2,3,2] + z[2,4,2] + z[2,5,2] + z[2,6,2] + z[2,7,2] + z[2,8,2] + z[2,9,2] + z[2,10,2] - 2 Z[2,2] >= 0
 z[3,1,1] + z[3,2,1] + z[3,3,1] + z[3,4,1] + z[3,5,1] + z[3,6,1] + z[3,7,1] + z[3,8,1] + z[3,9,1] + z[3,10,1] - 2 Z[3,1] >= 0     z[3,1,2] + z[3,2,2] + z[3,3,2] + z[3,4,2] + z[3,5,2] + z[3,6,2] + z[3,7,2] + z[3,8,2] + z[3,9,2] + z[3,10,2] - 2 Z[3,2] >= 0

In [17]:
@constraint(model, [d_=1:length(D_star), p=1:P], A_min[d_] <= A(p, D_star[d_]))

2×10 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 -0.4215602092181444 x[1,1] - 1.2646806276544333 x[2,1] - 1.6862408368725776 x[3,1] - 2.1078010460907217 x[4,1] - 0.8431204184362888 x[5,1] - 0.4215602092181444 x[6,1] - 2.1078010460907217 x[7,1] - 1.6862408368725776 x[8,1] - 0.3540879396744934 x[1,2] - 1.0622638190234805 x[2,2] - 1.4163517586979737 x[3,2] - 1.770439698372467 x[4,2] - 0.7081758793489868 x[5,2] - 0.3540879396744934 x[6,2] - 1.770439698372467 x[7,2] - 1.4163517586979737 x[8,2] - 0.25421165374626065 x[1,3] - 0.7626349612387817 x[2,3] - 1.0168466149850426 x[3,3] - 1.2710582687313035 x[4,3] - 0.5084233074925213 x[5,3] - 0.25421165374626065 x[6,3] - 1.2710582687313035 x[7,3] - 1.0168466149850426 x[8,3] - 0.0181322083962816 x[1,4] - 0.054396625188844805 x[2,4] - 0.0725288335851264 x[3,4] - 0.09066104198140845 x[4,4] - 0.0362644167925632 x[5,4] - 0.018132208396281

In [18]:
# Get ideal point
@objective(model, Max, A())
optimize!(model)
A_optimal = value(A())

Running HiGHS 1.13.0 (git hash: 1bce6d5c8): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
MIP has 198 rows; 264 cols; 11142 nonzeros; 262 integer variables (166 binary)
Coefficient ranges:
  Matrix  [2e-02, 2e+01]
  Cost    [9e-03, 2e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
186 rows, 155 cols, 10598 nonzeros  0s
18 rows, 62 cols, 374 nonzeros  0s
18 rows, 24 cols, 144 nonzeros  0s
Presolve reductions: rows 18(-180); columns 24(-240); nonzeros 144(-10998) 

Solving MIP model with:
   18 rows
   24 cols (0 binary, 22 integer, 0 implied int., 2 continuous, 0 domain fixed)
   144 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u =

253.54972670213607

In [19]:
@objective(model, Max, B())
optimize!(model)
B_optimal = value(B())

MIP has 198 rows; 264 cols; 11142 nonzeros; 262 integer variables (166 binary)
Coefficient ranges:
  Matrix  [2e-02, 2e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
166 rows, 243 cols, 9468 nonzeros  0s
102 rows, 174 cols, 5102 nonzeros  0s
Presolve reductions: rows 102(-96); columns 174(-90); nonzeros 5102(-6040) 
Objective function is integral with scale 1

Solving MIP model with:
   102 rows
   174 cols (90 binary, 84 integer, 0 implied int., 0 continuous, 0 domain fixed)
   5102 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds       

100.0

In [20]:
@objective(model, Max, C())
optimize!(model)
C_optimal = value(C())

MIP has 198 rows; 264 cols; 11142 nonzeros; 262 integer variables (166 binary)
Coefficient ranges:
  Matrix  [2e-02, 2e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [4e+00, 1e+02]
Presolving model
166 rows, 158 cols, 9383 nonzeros  0s
67 rows, 139 cols, 2989 nonzeros  0s
Presolve reductions: rows 67(-131); columns 139(-125); nonzeros 2989(-8153) 
Objective function is integral with scale 1

Solving MIP model with:
   67 rows
   139 cols (55 binary, 84 integer, 0 implied int., 0 continuous, 0 domain fixed)
   2989 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds        

6.0

In [21]:
ideal_point = [A_optimal, B_optimal, C_optimal]

3-element Vector{Float64}:
 253.54972670213607
 100.0
   6.0

In [22]:
A_norm = () -> A()/A_optimal
B_norm = () -> B()/B_optimal
C_norm = () -> C()/C_optimal

#91 (generic function with 1 method)

In [23]:
# Create an array of weights
weights_for_moa =   [[0.33, 0.33, 0.33],
                     [0.5, 0.25, 0.25],
                     [0.25, 0.5, 0.25],
                     [0.25, 0.25, 0.5],
                    ]

4-element Vector{Vector{Float64}}:
 [0.33, 0.33, 0.33]
 [0.5, 0.25, 0.25]
 [0.25, 0.5, 0.25]
 [0.25, 0.25, 0.5]

In [24]:
# Solve model with different weight distribution
set_silent(model)
set_optimizer_attribute(model, "mip_rel_gap", 0.1)

output_folder = "results"
solutions = []
solutions_nonrm = []
for i in 1:length(weights_for_moa)
    wA, wB, wC = weights_for_moa[i]
    @objective(model, Max, wA*A_norm()+wB*B_norm()+wC*C_norm())

    optimize!(model)

	solution = [value(A_norm()), value(B_norm()), value(C_norm())]
	solution_nonrm = [value(A()), value(B()), value(C())]
	push!(solutions, solution)
	push!(solutions_nonrm, solution_nonrm)

	output_file = "$(input_file_name)_$(weights_for_moa[i])"

	open(output_folder * "/" * output_file, "w") do io
		println(io, "Solution of model with weight distribution ", weights_for_moa[i])

		println(io, "Ideal Point (Bound): ", ideal_point)

		println(io, "---Итоговая эффективность тренировок---")
		println(io, "A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
		println(io, "A = $(value(A()))\nB = $(value(B()))\nC = $(value(C()))\n\n")
		println(io, "---Итоговая эффективность тренировок относительно идеальной точки---")
		println(io, "A = $(value(A_norm()))\nB = $(value(B_norm()))\nC = $(value(C_norm()))\n\n")

		println(io, "---Расписание тренировок---\n")
		n_days = length(D_)
		# Таблица: каждая строка — один день
		data = Array{Any}(undef, n_days, 2)
		for d_ in 1:n_days
			# 1-й столбец: информация о дне/тренировке
			data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"
			# Собираем все упражнения, попавшие в этот день
			day_exercises = String[]
			for e in 1:E
				if value(x[e, d_]) >= 1
					push!(day_exercises, Exercise_names[e])
				end
			end
			# 2-й столбец: список упражнений через запятую
			data[d_, 2] = join(day_exercises, ", ")
		end
		header = ["День", "Упражнения"]
		pretty_table(io, data, crop = :none)

		println(io, "---Освоенные навыки---\n")
		for p in 1:P
			for d_ in 1:length(D_star)	
				println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
				learned_anything = false
				for s in 1:S
					if (value(y[s, p, d_]) == 1 && (d_ == 1 || value(y[s, p, d_-1]) != 1.0))
						learned_anything = true
						println(io, "Освоил навык '$(Skill_names[s])'")
					end
				end
				if (learned_anything == false)
					println(io, "Ничего не освоил")
				end
			end
			print(io, "\n")
		end

		println(io, "---Освоенные индивидуально тактики---\n")
		for p in 1:P
			for d_ in 1:length(D_star)	
				println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
				learned_anything = false
				for t in 1:T
					if (value(z[t, p, d_]) == 1 && (d_ == 1 || value(z[t, p, d_-1]) != 1.0))
						learned_anything = true
						println(io, "Освоил тактику '$(Tactic_names[t])'")
					end
				end
				if (learned_anything == false)
					println(io, "Ничего не освоил")
				end
			end
			print(io, "\n")
		end

		println(io, "---Освоенные командой тактики---\n")
		for d_ in 1:length(D_star)	
			for t in 1:T	
				println(io, "Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
				learned_anything = false
				if value(Z[t, d_]) == 1
					println(io, "Освоено")
				else
					println(io, "Не освоено")
				end
			end
			print(io, "\n")
		end
	end
	
end



In [27]:
if !isempty(solutions_nonrm)
    A_vals = [sol[1] for sol in solutions_nonrm]
    B_vals = [sol[2] for sol in solutions_nonrm]
    C_vals = [sol[3] for sol in solutions_nonrm]

    ideal_A, ideal_B, ideal_C = ideal_point

    plt = scatter3d(A_vals, B_vals, C_vals, label="Solutions", xlabel="A", ylabel="B", zlabel="C", color=:blue)
    scatter3d!(plt, [ideal_A], [ideal_B], [ideal_C], label="Ideal Point", color=:red, markersize=10)

    # Animate the plot by rotating it
    anim = @animate for angle in 0:2:360
        plot!(plt, camera=(angle, 30))  # Adjust the camera angle
    end

    # Save the animation as a GIF (optional)
    gif(anim, output_folder * "/$(input_file_name)_solutions_nonrm.gif", fps=20)
else
    println("No solutions to visualize.")
end

AssertionError: AssertionError: total_plotarea_horizontal > 0mm

In [ ]:
if !isempty(solutions)
    A_vals = [sol[1] for sol in solutions]
    B_vals = [sol[2] for sol in solutions]
    C_vals = [sol[3] for sol in solutions]

    ideal_A, ideal_B, ideal_C = [1, 1, 1]

    plt = scatter3d(A_vals, B_vals, C_vals, label="Solutions", xlabel="A", ylabel="B", zlabel="C", color=:blue)
    scatter3d!(plt, [ideal_A], [ideal_B], [ideal_C], label="Ideal Point", color=:red, markersize=10)

    # Animate the plot by rotating it
    anim = @animate for angle in 0:2:360
        plot!(plt, camera=(angle, 30))  # Adjust the camera angle
    end

    # Save the animation as a GIF (optional)
    gif(anim, output_folder * "/$(input_file_name)_solutions_normalized.gif", fps=20)
else
    println("No solutions to visualize.")
end

AssertionError: AssertionError: total_plotarea_horizontal > 0mm